# 01 · Download BTS on-time data and GHCNh station files
This downloads the monthly zip files BTS publishes at
`transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{YYYY}_{M}.zip` (about 27 MB each),
keeps the needed columns, renames them to the TranStats field-picker names (`FL_DATE`, `CRS_DEP_TIME`, ...),
and saves `data/raw/bts/YYYY_MM.parquet` covering **all airports**.

You can rerun it at any time. Months that are already saved are skipped, and a month BTS hasn't published yet (HTTP 404) is reported and skipped.

In [1]:
%run ../config.ipynb

ROOT = /Users/dylannegvesky/Documents/FSU Documents/Fall 2026/ProjDS/PDS_Final
study airports: ['EWR'] | BTS 2023-01 → latest | train ≤ 2025-12


In [2]:
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm.auto import tqdm

BTS_URL = ("https://transtats.bts.gov/PREZIP/"
           "On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{y}_{m}.zip")
GHCNH_URL = ("https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/"
             "access/by-station/GHCNh_{station}_por.psv")

# BTS prezip column → TranStats field-picker name (matches the manual download in data/raw/bts_manual)
BTS_COLUMNS = {
    "Year": "YEAR", "Quarter": "QUARTER", "Month": "MONTH", "DayofMonth": "DAY_OF_MONTH",
    "DayOfWeek": "DAY_OF_WEEK", "FlightDate": "FL_DATE",
    "Reporting_Airline": "OP_UNIQUE_CARRIER", "Tail_Number": "TAIL_NUM",
    "Flight_Number_Reporting_Airline": "OP_CARRIER_FL_NUM",
    "Origin": "ORIGIN", "OriginCityName": "ORIGIN_CITY_NAME", "OriginStateName": "ORIGIN_STATE_NM",
    "Dest": "DEST", "DestCityName": "DEST_CITY_NAME", "DestStateName": "DEST_STATE_NM",
    "CRSDepTime": "CRS_DEP_TIME", "DepTime": "DEP_TIME", "DepDelay": "DEP_DELAY",
    "DepDelayMinutes": "DEP_DELAY_NEW", "DepDel15": "DEP_DEL15",
    "TaxiOut": "TAXI_OUT", "WheelsOff": "WHEELS_OFF", "WheelsOn": "WHEELS_ON", "TaxiIn": "TAXI_IN",
    "CRSArrTime": "CRS_ARR_TIME", "ArrTime": "ARR_TIME", "ArrDelay": "ARR_DELAY",
    "ArrDelayMinutes": "ARR_DELAY_NEW", "ArrDel15": "ARR_DEL15",
    "Cancelled": "CANCELLED", "CancellationCode": "CANCELLATION_CODE", "Diverted": "DIVERTED",
    "CRSElapsedTime": "CRS_ELAPSED_TIME", "ActualElapsedTime": "ACTUAL_ELAPSED_TIME",
    "AirTime": "AIR_TIME", "Distance": "DISTANCE",
    "CarrierDelay": "CARRIER_DELAY", "WeatherDelay": "WEATHER_DELAY", "NASDelay": "NAS_DELAY",
    "SecurityDelay": "SECURITY_DELAY", "LateAircraftDelay": "LATE_AIRCRAFT_DELAY",
}
# hhmm times stay strings so leading zeros survive ("0659")
BTS_STRING_COLS = ["Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline", "Origin", "Dest",
                   "CRSDepTime", "DepTime", "WheelsOff", "WheelsOn", "CRSArrTime", "ArrTime",
                   "CancellationCode", "OriginCityName", "OriginStateName", "DestCityName", "DestStateName"]


def month_list(start=BTS_START, end=BTS_END):
    last = pd.Period(end, "M") if end else pd.Timestamp.today().to_period("M") - 1
    return list(pd.period_range(pd.Period(start, "M"), last, freq="M"))


def download(url, dest, retries=3):
    """Stream url → dest. Returns HTTP status (404 means not published)."""
    part = dest.with_name(dest.name + ".part")
    for attempt in range(retries):
        try:
            with requests.get(url, stream=True, timeout=120) as r:
                if r.status_code == 404:
                    return 404
                r.raise_for_status()
                with open(part, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
            part.replace(dest)
            return 200
        except requests.RequestException as e:
            if attempt == retries - 1:
                raise
            time.sleep(5 * 2 ** attempt)


def fetch_bts_month(period):
    out = BTS_DIR / f"{period.year}_{period.month:02d}.parquet"
    if out.exists():
        return period, "skipped (exists)"
    zpath = BTS_DIR / f"{period.year}_{period.month:02d}.zip"
    status = download(BTS_URL.format(y=period.year, m=period.month), zpath)
    if status == 404:
        return period, "not published yet"
    with zipfile.ZipFile(zpath) as z:
        csv_name = next(n for n in z.namelist() if n.endswith(".csv"))
        with z.open(csv_name) as f:
            df = pd.read_csv(f, usecols=list(BTS_COLUMNS),
                             dtype={c: "string" for c in BTS_STRING_COLS}, low_memory=False)
    df = df.rename(columns=BTS_COLUMNS)
    missing = set(BTS_COLUMNS.values()) - set(df.columns)
    assert not missing, f"{period}: missing columns {missing}"
    df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])
    tmp = out.with_name(out.name + ".part")
    df.to_parquet(tmp, index=False)
    tmp.replace(out)
    zpath.unlink()
    return period, f"saved {len(df):,} rows"

/Users/dylannegvesky/Documents/FSU Documents/Fall 2026/ProjDS/PDS_Final/.venv/lib/python3.15/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## BTS months
Up to 3 months download in parallel. The progress bar counts months.

In [3]:
months = month_list()
print(f"{len(months)} months: {months[0]} → {months[-1]}")
results = {}
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = [pool.submit(fetch_bts_month, p) for p in months]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="BTS months"):
        p, msg = fut.result()
        results[p] = msg
for p in sorted(results):
    print(p, "—", results[p])

46 months: 2023-01 → 2026-10


BTS months:   0%|          | 0/46 [00:00<?, ?it/s]

BTS months: 100%|██████████| 46/46 [00:00<00:00, 128.72it/s]

BTS months: 100%|██████████| 46/46 [00:00<00:00, 128.52it/s]

2023-01 — skipped (exists)
2023-02 — skipped (exists)
2023-03 — skipped (exists)
2023-04 — skipped (exists)
2023-05 — skipped (exists)
2023-06 — skipped (exists)
2023-07 — skipped (exists)
2023-08 — skipped (exists)
2023-09 — skipped (exists)
2023-10 — skipped (exists)
2023-11 — skipped (exists)
2023-12 — skipped (exists)
2024-01 — skipped (exists)
2024-02 — skipped (exists)
2024-03 — skipped (exists)
2024-04 — skipped (exists)
2024-05 — skipped (exists)
2024-06 — skipped (exists)
2024-07 — skipped (exists)
2024-08 — skipped (exists)
2024-09 — skipped (exists)
2024-10 — skipped (exists)
2024-11 — skipped (exists)
2024-12 — skipped (exists)
2025-01 — skipped (exists)
2025-02 — skipped (exists)
2025-03 — skipped (exists)
2025-04 — skipped (exists)
2025-05 — skipped (exists)
2025-06 — skipped (exists)
2025-07 — skipped (exists)
2025-08 — skipped (exists)
2025-09 — skipped (exists)
2025-10 — skipped (exists)
2025-11 — skipped (exists)
2025-12 — skipped (exists)
2026-01 — skipped (exists)
2

## GHCNh station files for the study airports
Each file is a station's full history (Newark's is about 800 MB). Files already present are skipped.

In [4]:
for apt, cfg in STUDY_AIRPORTS.items():
    st = cfg["ghcnh_station"]
    dest = GHCNH_DIR / f"GHCNh_{st}_por.psv"
    if dest.exists():
        print(f"{apt}: {dest.name} already present ({dest.stat().st_size/1e6:,.0f} MB)")
        continue
    print(f"{apt}: downloading {dest.name} …")
    status = download(GHCNH_URL.format(station=st), dest)
    print(f"{apt}: HTTP {status}")

EWR: GHCNh_USW00014734_por.psv already present (824 MB)


## Check

In [5]:
saved = sorted(BTS_DIR.glob("*.parquet"))
print(f"{len(saved)} BTS month files, {sum(p.stat().st_size for p in saved)/1e9:.2f} GB")
expected = {f"{p.year}_{p.month:02d}.parquet" for p in months if results.get(p) != "not published yet"}
missing = expected - {p.name for p in saved}
assert not missing, f"missing months: {sorted(missing)}"
pd.read_parquet(saved[-1]).head()

45 BTS month files, 0.67 GB


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_NM,DEST,DEST_CITY_NAME,DEST_STATE_NM,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,DEP_DELAY_NEW,DEP_DEL15,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,ARR_DELAY_NEW,ARR_DEL15,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ACTUAL_ELAPSED_TIME,AIR_TIME,DISTANCE,CARRIER_DELAY,WEATHER_DELAY,NAS_DELAY,SECURITY_DELAY,LATE_AIRCRAFT_DELAY
0,2026,3,9,9,3,2026-09-09,DL,N693DL,1372,MCO,"Orlando, FL",Florida,ATL,"Atlanta, GA",Georgia,1640,1632,-8.0,0.0,0.0,13.0,1645,1747,6.0,1825,1753,-32.0,0.0,0.0,0.0,<NA>,0.0,105.0,81.0,62.0,404.0,NaN,NaN,NaN,NaN,NaN
1,2026,3,9,10,4,2026-09-10,DL,N699DL,1372,ATL,"Atlanta, GA",Georgia,MCO,"Orlando, FL",Florida,1400,1357,-3.0,0.0,0.0,11.0,1408,1508,7.0,1530,1515,-15.0,0.0,0.0,0.0,<NA>,0.0,90.0,78.0,60.0,404.0,NaN,NaN,NaN,NaN,NaN
2,2026,3,9,10,4,2026-09-10,DL,N699DL,1372,MCO,"Orlando, FL",Florida,ATL,"Atlanta, GA",Georgia,1640,1641,1.0,1.0,0.0,46.0,1727,1838,7.0,1825,1845,20.0,20.0,1.0,0.0,<NA>,0.0,105.0,124.0,71.0,404.0,0.0,0.0,20.0,0.0,0.0
3,2026,3,9,11,5,2026-09-11,DL,N6702,1372,ATL,"Atlanta, GA",Georgia,MCO,"Orlando, FL",Florida,1400,1401,1.0,1.0,0.0,27.0,1428,1531,8.0,1530,1539,9.0,9.0,0.0,0.0,<NA>,0.0,90.0,98.0,63.0,404.0,NaN,NaN,NaN,NaN,NaN
4,2026,3,9,11,5,2026-09-11,DL,N6702,1372,MCO,"Orlando, FL",Florida,ATL,"Atlanta, GA",Georgia,1640,1656,16.0,16.0,1.0,31.0,1727,1839,5.0,1825,1844,19.0,19.0,1.0,0.0,<NA>,0.0,105.0,108.0,72.0,404.0,0.0,0.0,19.0,0.0,0.0
